# Classificação de doenças em folhas de arroz (Paddy Doctor)

Este notebook treina e compara seis redes neurais convolucionais na tarefa de reconhecer **10 classes** de imagens de folhas de arroz
(9 doenças + folhas saudáveis), usando o dataset *Paddy Doctor* (versão da competição *Paddy Disease
Classification* do Kaggle, com 10.407 imagens).

A ideia geral é:

1. Carregar o dataset, com as imagens organizadas em pastas (uma pasta por classe).
2. Dividir as imagens em **treino**, **validação** e **teste**.
3. Partir de modelos já treinados no ImageNet (*transfer learning*) e ajustá-los para as nossas classes.
4. Avaliar cada modelo no conjunto de teste e gerar gráficos e métricas.

> Recomenda-se rodar em um ambiente com **GPU**. O código também funciona na CPU, mas fica bem mais lento.

## 1. Preparação do ambiente

Instala as bibliotecas usadas no projeto. O endereço extra (`--extra-index-url`) é o repositório oficial
do PyTorch com as versões compiladas para **CUDA 13.0**, para que o treino use a GPU.

In [ ]:
!pip install -q --extra-index-url https://download.pytorch.org/whl/cu130 timm torch torchvision torchaudio scikit-learn matplotlib seaborn pandas numpy pillow

## 2. Carregamento do dataset

As imagens ficam em um arquivo `datasets.zip` no Google Drive. Dentro do zip há a pasta `datasets/`,
com uma subpasta por classe:

```
datasets/
├── bacterial_leaf_blight/
├── bacterial_leaf_streak/
├── ...
└── tungro/
```

A célula abaixo conecta o Google Drive (é preciso autorizar o acesso na primeira vez),
copia o zip para o disco local e o extrai. O Drive também é usado no fim de cada treino,
para guardar uma cópia dos resultados. Trabalhar com uma cópia local deixa a leitura das imagens
durante o treino bem mais rápida do que ler direto do Drive.

Se o zip estiver em outra pasta do Drive, ajuste `DRIVE_ZIP`. Se a pasta `datasets/` já existir,
a extração é pulada.

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive

DRIVE_ZIP = Path("/content/drive/MyDrive/datasets.zip")

drive.mount("/content/drive")

if not Path("datasets").exists():
    shutil.copy(DRIVE_ZIP, "datasets.zip")
    shutil.unpack_archive("datasets.zip", ".")

for class_dir in sorted(Path("datasets").iterdir()):
    print(f"{class_dir.name}: {len(list(class_dir.iterdir()))} imagens")

## 3. Bibliotecas

- **PyTorch / torchvision**: construção, treino e modelos pré-treinados.
- **scikit-learn**: divisão estratificada dos dados e métricas de avaliação.
- **pandas / numpy**: manipulação de tabelas e vetores.
- **matplotlib / seaborn**: gráficos. Eles são salvos como arquivos PNG e exibidos no final do notebook.

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

## 4. Configurações

Todos os parâmetros do experimento ficam aqui. Os mais importantes:

| Parâmetro | O que faz |
|---|---|
| `MODEL_NAME` | Arquitetura usada (`mobilenet_v3_small`, `mobilenet_v3_large`, `resnet18`, `resnet50`, `convnext_tiny`, `convnext_large`). |
| `EPOCHS` | Número máximo de passagens completas pelos dados de treino (50). |
| `EARLY_STOP_PATIENCE` | Para o treino se a validação não melhorar por esse número de épocas (10). |
| `USE_AUGMENTATION` | Aplica transformações aleatórias nas imagens de treino (giro, espelhamento, cor). |
| `USE_CLASS_WEIGHTS` | Dá mais peso às classes com menos imagens, compensando o desbalanceamento. |
| `FREEZE_EPOCHS` | Nas primeiras épocas só a última camada treina; depois, o modelo inteiro. |
| `TRAIN_RATIO`, `VALID_RATIO`, `TEST_RATIO` | Proporção de imagens em cada conjunto (70% / 15% / 15%). |

`IMAGENET_MEAN` e `IMAGENET_STD` são a média e o desvio padrão usados para normalizar as imagens,
os mesmos do treino original dos modelos no ImageNet.

A seção 14 treina os seis modelos em sequência, trocando `MODEL_NAME` a cada rodada.

In [ ]:
MODEL_NAME = "resnet50"  # modelos disponiveis: "resnet18", "resnet50", "convnext_tiny", "convnext_large", "mobilenet_v3_small", "mobilenet_v3_large"
DATASET_DIR = Path("datasets/")
RESULTS_DIR = Path("results")

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 4
EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 10

USE_AUGMENTATION = True
USE_CLASS_WEIGHTS = True # compensa o desbalanceamento das classes na loss
USE_CUDA = True # usa a GPU (CUDA) quando disponivel; senao cai para a CPU
USE_AMP = True # precisao mista (float16) na GPU: treino mais rapido e com menos memoria
FREEZE_EPOCHS = 3 # fine-tuning em 2 etapas: primeiro so a cabeca, depois o modelo todo (0 desliga)
FINETUNE_LR = 1e-4 # taxa de aprendizado da 2a etapa (modelo completo)

TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

N_EXAMPLES_QUALITATIVE = 5 # numero de exemplos certos/errados na analise qualitativa

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

## 5. Funções auxiliares

- `set_seed`: fixa a semente aleatória para que os resultados possam ser reproduzidos.
- `get_device`: usa a GPU (CUDA) se houver uma; senão, a CPU.
- `amp_enabled`: liga a precisão mista (float16), que acelera o treino na GPU e gasta menos memória.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    if USE_CUDA and torch.cuda.is_available():
        print(f"CUDA disponivel: {torch.cuda.get_device_name(0)}")
        torch.backends.cudnn.benchmark = True # escolhe os kernels mais rapidos para o tamanho fixo da entrada
        return torch.device("cuda")
    print("CUDA indisponivel, usando CPU")
    return torch.device("cpu")


def amp_enabled(device):
    return USE_AMP and device.type == "cuda"

## 6. Pré-processamento e aumento de dados

`get_transforms` define o que acontece com cada imagem antes de entrar na rede:

- **Treino** (com aumento de dados): recorte aleatório, espelhamento horizontal e vertical, rotação de até 30°
  e pequenas variações de brilho, contraste, saturação e tom. Assim o modelo vê versões um pouco diferentes
  da mesma imagem a cada época, o que reduz o *overfitting*.
- **Validação e teste**: só redimensiona e recorta o centro em 224×224, sem nada aleatório.

Em todos os casos a imagem vira um tensor e é normalizada.

`SubsetWithTransform` serve para aplicar transformações diferentes a cada parte da divisão
(treino, validação e teste), mesmo que todas venham do mesmo dataset.

In [ ]:
def get_transforms(train):
    if train and USE_AUGMENTATION:
        return transforms.Compose([
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(30),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class SubsetWithTransform(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        image, label = self.subset[index]
        if self.transform:
            image = self.transform(image)
        return image, label

## 7. Modelos

`build_model` carrega a arquitetura escolhida com pesos pré-treinados no ImageNet e troca a última camada
(a "cabeça" do classificador) por uma nova com uma saída para cada uma das nossas 10 classes.

Esse é o *transfer learning*: o modelo já sabe reconhecer bordas, texturas e formas, e só precisa aprender
a usar isso para distinguir as doenças.

In [ ]:
def build_model(name, num_classes):
    if name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    elif name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "convnext_tiny":
        model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    elif name == "convnext_large":
        model = models.convnext_large(weights=models.ConvNeXt_Large_Weights.IMAGENET1K_V1)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    elif name == "mobilenet_v3_large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    else:
        raise ValueError(f"Modelo '{name}' nao suportado.")
    return model

## 8. Carregamento em lotes

O `DataLoader` entrega as imagens à rede em lotes de `BATCH_SIZE` imagens, usando `NUM_WORKERS` processos
em paralelo para ler os arquivos do disco.

In [ ]:
def make_dataloader(dataset, shuffle):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0, # nao recria os workers a cada epoca
    )

## 9. Uma época de treino e avaliação

- `train_one_epoch`: passa por todos os lotes de treino. Para cada lote, calcula a previsão, mede o erro
  (*loss*) e ajusta os pesos da rede. No fim, devolve a loss média e a acurácia.
- `evaluate_loader`: faz o mesmo cálculo, mas **sem** ajustar os pesos. É usada para medir o desempenho na validação.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=scaler.is_enabled()):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()
        total_samples += images.size(0)
    return total_loss / total_samples, total_correct / total_samples


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=amp_enabled(device)):
            outputs = model(images)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        total_correct += (outputs.argmax(1) == labels).sum().item()
        total_samples += images.size(0)
    return total_loss / total_samples, total_correct / total_samples

## 10. Fine-tuning em duas etapas e otimizador

- `freeze_backbone`: "congela" o corpo da rede, deixando só a cabeça treinar. Com `freeze=False`, libera o modelo inteiro.
- `make_optimizer`: cria o otimizador **AdamW** e um *scheduler* que divide a taxa de aprendizado por 2
  quando a loss de validação para de cair por 2 épocas.

Treinar primeiro só a cabeça evita que a camada nova, ainda com pesos aleatórios,
estrague o que o modelo pré-treinado já sabia.

In [ ]:
def freeze_backbone(model, name, freeze):
    # Congelado: so a ultima camada (cabeca) treina. Descongelado: o modelo inteiro treina.
    head_prefix = "fc." if name.startswith("resnet") else "classifier."
    for param_name, param in model.named_parameters():
        param.requires_grad = (not freeze) or param_name.startswith(head_prefix)


def make_optimizer(model, lr):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    return optimizer, scheduler

## 11. Laço de treino

`train_model` junta as peças anteriores:

1. **Etapa 1** (`FREEZE_EPOCHS` épocas): só a cabeça treina, com `LEARNING_RATE`.
2. **Etapa 2**: o modelo inteiro treina, com uma taxa menor (`FINETUNE_LR`).
3. A cada época, avalia na validação e guarda o histórico de loss e acurácia.
4. Sempre que a loss de validação melhora, salva o modelo em `best_model.pth`.
5. **Early stopping**: se a validação não melhorar por `EARLY_STOP_PATIENCE` épocas seguidas, o treino para.

In [ ]:
def train_model(model, train_loader, valid_loader, criterion, device, save_dir):
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled(device))

    if FREEZE_EPOCHS > 0:
        freeze_backbone(model, MODEL_NAME, freeze=True)
        print(f"--- Etapa 1: so a cabeca, por {FREEZE_EPOCHS} epocas ---")
    optimizer, scheduler = make_optimizer(model, LEARNING_RATE)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS + 1:
            freeze_backbone(model, MODEL_NAME, freeze=False)
            optimizer, scheduler = make_optimizer(model, FINETUNE_LR)
            print(f"--- Etapa 2: modelo completo, lr={FINETUNE_LR} ---")

        start = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
        val_loss, val_acc = evaluate_loader(model, valid_loader, criterion, device)
        scheduler.step(val_loss)
        elapsed = time.time() - start

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoca {epoch:02d}/{EPOCHS} | "
              f"train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} | "
              f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f} | tempo: {elapsed:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), save_dir / "best_model.pth")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                print(f"Early stopping na epoca {epoch}")
                break

    return history

## 12. Gráficos e análise dos resultados

- `plot_history`: curvas de loss e acurácia ao longo das épocas, em treino e validação.
  Se a curva de validação se afasta muito da de treino, é sinal de *overfitting*.
- `get_predictions`: roda o modelo no conjunto de teste e guarda as classes reais e as previstas.
- `plot_confusion_matrix`: mostra quantas imagens de cada classe real foram previstas como cada classe.
  A diagonal são os acertos; fora dela, as confusões entre doenças.
- `qualitative_analysis`: mostra alguns exemplos que o modelo acertou (verde) e alguns que errou (vermelho).

In [ ]:
def plot_history(history, save_path):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, history["train_loss"], label="Train")
    ax1.plot(epochs, history["val_loss"], label="Validation")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()

    ax2.plot(epochs, history["train_acc"], label="Train")
    ax2.plot(epochs, history["val_acc"], label="Validation")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.legend()

    fig.tight_layout()
    fig.savefig(save_path)
    plt.close(fig)

In [ ]:
@torch.no_grad()
def get_predictions(model, test_loader, device):
    model.eval()
    y_true, y_pred = [], []
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        y_true.extend(labels.tolist())
        y_pred.extend(outputs.argmax(1).cpu().tolist())
    return np.array(y_true), np.array(y_pred)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(f"Matriz de Confusao - {MODEL_NAME}")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    fig.tight_layout()
    fig.savefig(save_path)
    plt.close(fig)

In [ ]:
def qualitative_analysis(model, test_dataset, class_names, device, save_path):
    examples_correct, examples_wrong = [], []

    indices = list(range(len(test_dataset)))
    for idx in indices:
        image, label = test_dataset[idx]
        with torch.no_grad():
            pred = model(image.unsqueeze(0).to(device)).argmax(1).item()
        if pred == label:
            examples_correct.append((image, label, pred))
        else:
            examples_wrong.append((image, label, pred))
        if len(examples_correct) >= N_EXAMPLES_QUALITATIVE and len(examples_wrong) >= N_EXAMPLES_QUALITATIVE:
            break

    n_correct = min(N_EXAMPLES_QUALITATIVE, len(examples_correct))
    n_wrong = min(N_EXAMPLES_QUALITATIVE, len(examples_wrong))
    fig, axes = plt.subplots(2, N_EXAMPLES_QUALITATIVE, figsize=(2.4 * N_EXAMPLES_QUALITATIVE, 5))

    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

    for col, (image, label, pred) in enumerate(examples_correct[:N_EXAMPLES_QUALITATIVE]):
        img = (image * std + mean).permute(1, 2, 0).numpy()
        axes[0, col].imshow(np.clip(img, 0, 1))
        axes[0, col].set_title(f"Real: {class_names[label]}\nPredito: {class_names[pred]}", color="green", fontsize=9)
        axes[0, col].axis("off")

    for col, (image, label, pred) in enumerate(examples_wrong[:N_EXAMPLES_QUALITATIVE]):
        img = (image * std + mean).permute(1, 2, 0).numpy()
        axes[1, col].imshow(np.clip(img, 0, 1))
        axes[1, col].set_title(f"Real: {class_names[label]}\nPredito: {class_names[pred]}", color="red", fontsize=9)
        axes[1, col].axis("off")

    axes[0, 0].set_ylabel("Corretas", fontsize=10)
    axes[1, 0].set_ylabel("Incorretas", fontsize=10)
    fig.suptitle(f"Analise Qualitativa - {MODEL_NAME} (topo: acertos, baixo: erros)")
    fig.tight_layout()
    fig.savefig(save_path)
    plt.close(fig)

## 13. Função principal

`main` executa o experimento completo, na ordem:

1. Fixa a semente e escolhe o dispositivo (GPU ou CPU).
2. Lê as imagens de `datasets/` (cada pasta é uma classe) e mostra quantas imagens há por classe.
3. Divide os dados em treino, validação e teste de forma **estratificada**, ou seja, mantendo a proporção
   de cada classe nos três conjuntos. A divisão é salva em `results/split.csv`.
4. Cria o modelo e a função de perda (*cross-entropy*), com pesos por classe se `USE_CLASS_WEIGHTS` estiver ligado.
5. Treina o modelo e salva o histórico (`history.csv` e `history.png`).
6. Recarrega o melhor modelo salvo e avalia **apenas no conjunto de teste**, calculando acurácia,
   precisão, recall e F1-score.
   - **macro**: média simples entre as classes (todas pesam igual).
   - **weighted**: média ponderada pelo número de imagens de cada classe.
7. Salva o relatório por classe, a matriz de confusão, a análise qualitativa e uma linha com as métricas
   em `results/metrics_summary.csv` (uma linha por modelo, para comparar arquiteturas).

In [ ]:
def main():
    set_seed(SEED)
    device = get_device()

    print(f"Dispositivo: {device}")
    print(f"Modelo: {MODEL_NAME}")

    save_dir = RESULTS_DIR / MODEL_NAME
    save_dir.mkdir(parents=True, exist_ok=True)

    # Carregar o dataset e dividir em treino/validacao/teste
    
    full_dataset = datasets.ImageFolder(DATASET_DIR)
    class_names = full_dataset.classes
    num_classes = len(class_names)
    print(f"\nClasses ({num_classes}): {class_names}")

    labels = np.array(full_dataset.targets)
    indices = np.arange(len(labels))
    counts = pd.Series(labels).value_counts().sort_index()
    print("\nDistribuicao das classes no dataset completo:")
    for i, count in enumerate(counts):
        print(f"  {class_names[i]}: {count}")

    train_idx, temp_idx = train_test_split(
        indices, test_size=VALID_RATIO + TEST_RATIO,
        stratify=labels, random_state=SEED)
    valid_idx, test_idx = train_test_split(
        temp_idx, test_size=TEST_RATIO / (VALID_RATIO + TEST_RATIO),
        stratify=labels[temp_idx], random_state=SEED)

    split_df = pd.DataFrame({
        "path": [full_dataset.samples[i][0] for i in indices],
        "label": [class_names[y] for y in labels],
        "split": ["train" if i in train_idx else "valid" if i in valid_idx else "test" for i in indices],
    })
    split_df.to_csv(RESULTS_DIR / "split.csv", index=False)

    print(f"\nDivisao (seed={SEED}, estratificada): "
          f"treino={len(train_idx)}, validacao={len(valid_idx)}, teste={len(test_idx)}")

    train_dataset = SubsetWithTransform(Subset(full_dataset, train_idx), get_transforms(train=True))
    valid_dataset = SubsetWithTransform(Subset(full_dataset, valid_idx), get_transforms(train=False))
    test_dataset = SubsetWithTransform(Subset(full_dataset, test_idx), get_transforms(train=False))

    train_loader = make_dataloader(train_dataset, shuffle=True)
    valid_loader = make_dataloader(valid_dataset, shuffle=False)
    test_loader = make_dataloader(test_dataset, shuffle=False)

    # Criar o modelo e a funcao de perda

    model = build_model(MODEL_NAME, num_classes).to(device).to(memory_format=torch.channels_last)

    if USE_CLASS_WEIGHTS:
        train_labels = [full_dataset.targets[i] for i in train_idx]
        weights = 1.0 / np.bincount(train_labels)
        weights = weights / weights.sum() * num_classes
        class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
        print("\nPesos das classes na loss:", np.round(weights, 3))
    else:
        class_weights = None

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Treinar com validacao e early stopping

    print(f"\n--- Treinando {MODEL_NAME} ---")
    history = train_model(model, train_loader, valid_loader, criterion, device, save_dir)
    pd.DataFrame(history).to_csv(save_dir / "history.csv", index=False)
    plot_history(history, save_dir / "history.png")


    # Avaliar apenas no conjunto de teste
    model.load_state_dict(torch.load(save_dir / "best_model.pth", weights_only=True))
    y_true, y_pred = get_predictions(model, test_loader, device)

    accuracy = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    report = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)

    print(f"\n--- Resultados no conjunto de teste ({MODEL_NAME}) ---")
    print(f"Acuracia: {accuracy:.4f}")
    print(f"Precisao (macro): {precision_macro:.4f} | Precisao (weighted): {precision_weighted:.4f}")
    print(f"Recall (macro): {recall_macro:.4f} | Recall (weighted): {recall_weighted:.4f}")
    print(f"F1-score (macro): {f1_macro:.4f} | F1-score (weighted): {f1_weighted:.4f}")
    print("\nRelatorio por classe:")
    print(report)

    with open(save_dir / "classification_report.txt", "w") as f:
        f.write(f"Modelo: {MODEL_NAME}\n")
        f.write(f"Estrategia de agregacao das metricas: macro e weighted "
                f"(weighted e a mais adequada, pois as classes sao desbalanceadas)\n\n")
        f.write(report)

    plot_confusion_matrix(y_true, y_pred, class_names, save_dir / "confusion_matrix.png")
    qualitative_analysis(model, test_dataset, class_names, device, save_dir / "qualitative.png")

    summary = pd.DataFrame([{
        "model": MODEL_NAME,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "precision_weighted": precision_weighted,
        "recall_macro": recall_macro,
        "recall_weighted": recall_weighted,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }])
    summary_path = RESULTS_DIR / "metrics_summary.csv"
    if summary_path.exists():
        # remove a linha antiga deste modelo para nao duplicar ao rodar de novo
        old = pd.read_csv(summary_path)
        summary = pd.concat([old[old["model"] != MODEL_NAME], summary], ignore_index=True)
    summary.to_csv(summary_path, index=False)

    print(f"\nResultados salvos em: {save_dir}")

## 14. Executar

Esta célula treina e avalia os seis modelos, um depois do outro, com as mesmas configurações.
A cada rodada, `MODEL_NAME` recebe o próximo modelo da lista e `main()` executa o experimento completo.
O progresso de cada época aparece logo abaixo.

Ao fim de cada modelo, a pasta `results/` é copiada para `results/` no Google Drive. Assim, se o ambiente
de execução cair no meio, os modelos já concluídos não se perdem.

A ConvNeXt-Large é a última da lista por ser a mais lenta. Para treinar só alguns modelos, basta
remover os outros da lista.

In [ ]:
for MODEL_NAME in ["resnet18", "resnet50", "mobilenet_v3_small",
                   "mobilenet_v3_large", "convnext_tiny", "convnext_large"]:
    main()
    shutil.copytree(RESULTS_DIR, "/content/drive/MyDrive/results", dirs_exist_ok=True)

## 15. Visualizar os resultados

Os arquivos gerados ficam em `results/<MODEL_NAME>/`, uma pasta por modelo. A primeira célula abaixo mostra
os gráficos e o relatório do último modelo treinado; para ver outro, troque `MODEL_NAME` antes de executá-la.

In [ ]:
from IPython.display import Image, display

save_dir = RESULTS_DIR / MODEL_NAME
print((save_dir / "classification_report.txt").read_text())

for name in ["history.png", "confusion_matrix.png", "qualitative.png"]:
    display(Image(filename=str(save_dir / name)))

Comparação entre todos os modelos treinados (uma linha por modelo):

In [ ]:
pd.read_csv(RESULTS_DIR / "metrics_summary.csv")

## 16. Baixar os resultados (opcional)

Compacta a pasta `results/` em um arquivo `.zip` para guardar os resultados.

In [ ]:
shutil.make_archive("results", "zip", RESULTS_DIR)
print("Arquivo gerado: results.zip")